In [0]:
# ── CÉLULA 1: caminhos ───────────────────────────────────────────────
BRONZE_PATH = "/FileStore/lakehouse/bronze/"

# ── CÉLULA 2: leitura via Catalog (sem JDBC manual) ──────────────────
df_clientes = spark.table("postgresrender_catalog.public.clientes")
df_produtos  = spark.table("postgresrender_catalog.public.produtos")
df_pedidos   = spark.table("postgresrender_catalog.public.pedidos")

print(f"Clientes: {df_clientes.count()}")
print(f"Produtos: {df_produtos.count()}")
print(f"Pedidos:  {df_pedidos.count()}")

In [0]:
# ── CÉLULA 1: criar os volumes ───────────────────────────────────────
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.silver")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.gold")

print("Volumes criados!")

In [0]:
# ── CÉLULA 2: configuração ───────────────────────────────────────────
BRONZE_PATH = "/Volumes/workspace/default/bronze/"
SILVER_PATH = "/Volumes/workspace/default/silver/"
GOLD_PATH   = "/Volumes/workspace/default/gold/"

print("Config OK")

In [0]:
# ── CÉLULA 3: leitura do Postgres via Catalog ────────────────────────
df_clientes = spark.table("postgresrender_catalog.public.clientes")
df_produtos  = spark.table("postgresrender_catalog.public.produtos")
df_pedidos   = spark.table("postgresrender_catalog.public.pedidos")

print(f"Clientes: {df_clientes.count()}")
print(f"Produtos: {df_produtos.count()}")
print(f"Pedidos:  {df_pedidos.count()}")

In [0]:
# ── CÉLULA 4: escrita Bronze em Delta ────────────────────────────────
from pyspark.sql.functions import current_timestamp, lit

def escreve_bronze(df, nome_tabela):
    df.withColumn("_bronze_ingestao", current_timestamp()) \
      .withColumn("_origem", lit("postgres_render")) \
      .write \
      .format("delta") \
      .mode("overwrite") \
      .save(f"{BRONZE_PATH}{nome_tabela}/")
    print(f"Bronze {nome_tabela}: OK")

escreve_bronze(df_clientes, "clientes")
escreve_bronze(df_produtos,  "produtos")
escreve_bronze(df_pedidos,   "pedidos")

In [0]:
# ── CÉLULA 5: valida arquivos gravados ───────────────────────────────
display(dbutils.fs.ls(BRONZE_PATH))

In [0]:
# ── CÉLULA 6: confirma lendo de volta o Delta ────────────────────────
df_bronze_pedidos = spark.read.format("delta").load(f"{BRONZE_PATH}pedidos/")
display(df_bronze_pedidos)